# Configuration

In [1]:
import random
import pandas as pd 
import numpy as np

from tqdm import tqdm

In [2]:
from scipy.stats import norm

# estimators
from econml.grf import CausalForest

In [3]:
# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Estimators

In [4]:
# naive estimator
def naive_estimator(data: list):
    """ 
    maximum over the average of the variables
    """
    return max([sample.mean() for sample in data])

# bootstrap estimator
def bootstrap_estimator(data: list, num_bootstraps: int = 5000):
    """
    bootstrap the estimator
    """
    # bootstrapping
    bootstraps = pd.DataFrame({f'var{idx}': np.random.choice(value, size=num_bootstraps) for idx, value in enumerate(data)})
    return bootstraps.max(axis=1).mean()


# Estimating the expected maximum of random variables

## Independent variables

In [5]:
class RandomVariable(object):
    def sample(self, size: int):
        raise NotImplementedError()
    
    def cdf(self, x: float):
        """ 
        Cumulative distribution function
        """
        raise NotImplementedError()

class ContinuousRandomVariable(RandomVariable): 
    def pdf(self, x: float):
        """
        Probability density function (might not exist for some distributions)
        """
        raise NotImplementedError()
    
class DiscreteRandomVariable(RandomVariable): 
    def pmf(self, x: float):
        """ 
        Probability mass function
        """
        raise NotImplementedError()
    
class UnivariateGaussian(ContinuousRandomVariable):
    def __init__(self, mean: float, std: float):
        self.mean = mean
        self.std = std
    
    def sample(self, size: int):
        return np.random.normal(self.mean, self.std, size)
    
    def pdf(self, x: float):
        return norm.pdf(x, self.mean, self.std)

    def cdf(self, x: float):
        return norm.cdf(x, self.mean, self.std)
    
class UnivariateExponential(ContinuousRandomVariable):
    def __init__(self, rate: float):
        self.rate = rate
    
    def sample(self, size: int):
        return np.random.exponential(1 / self.rate, size)
    
    def pdf(self, x: float):
        return self.rate * np.exp(-self.rate * x)
    
    def cdf(self, x: float):
        return 1 - np.exp(-self.rate * x)
    

class Bernoulli(DiscreteRandomVariable):
    def __init__(self, p: float):
        self.p = p
    
    def sample(self, size: int):
        return np.random.binomial(1, self.p, size)
    
    def pmf(self, x: float):        
        if x == 1:
            return self.p
        elif x == 0:
            return 1 - self.p
        else:
            return 0
    
    def cdf(self, x: float):
        if x < 0:
            return 0
        elif x == 0:
            return 1 - self.p
        elif x < 1:
            return 1 - self.p
        else: 
            return 1

In [6]:
random_variable_meta = [
    {
        'variable': UnivariateGaussian(mean=3, std=2),
        'sample_size': 100
    }, 
    {
        'variable': UnivariateGaussian(mean=2, std=1),
        'sample_size': 200
    }, 
    {
        'variable': UnivariateGaussian(mean=1, std=10),
        'sample_size': 100
    }, 
    {
        'variable': UnivariateExponential(rate=1),
        'sample_size': 500
    }, 
    {
        'variable': UnivariateExponential(rate=0.5), 
        'sample_size': 50
    }, 
]


# sample from meta
data = [dict_['variable'].sample(dict_['sample_size']) for dict_ in random_variable_meta]

# calculate the mean 
def calculate_mean(meta: dict, sample_size: int = 500000) -> float: 
    sample_arr = np.zeros(sample_size)

    for i in range(sample_size):
        sample_arr[i] = max([var['variable'].sample(1)[0] for var in meta])

    return np.mean(sample_arr)
true_mean = calculate_mean(random_variable_meta)

In [83]:
print(f"Naive estimator: {naive_estimator(data)}")
print(f"Bootstrap estimator: {bootstrap_estimator(data)}")
print('True value: ', true_mean)

num_repeats = 100 
naive_estimates_arr = np.zeros(num_repeats)
bootstrap_estimates_arr = np.zeros(num_repeats)

for repeat_idx in range(num_repeats):
    # sample from meta
    data = [dict_['variable'].sample(dict_['sample_size']) for dict_ in random_variable_meta]

    # calculate the mean 
    naive_estimates_arr[repeat_idx] = naive_estimator(data)
    bootstrap_est = bootstrap_estimator(data)

# plot the results
plt.figure(figsize=(10, 6.18))
# box plot
sns.boxplot(data=[naive_estimates_arr, bootstrap_estimates_arr])

: 

## Dependent variables

In [77]:
random_variable_meta = [
    {
        'variable': UnivariateGaussian(mean=3, std=2),
        'sample_size': 100
    }, 
    {
        'variable': UnivariateGaussian(mean=2, std=1),
        'sample_size': 200
    }, 
    {
        'variable': UnivariateGaussian(mean=1, std=10),
        'sample_size': 100
    }, 
    {
        'variable': UnivariateExponential(rate=1),
        'sample_size': 500
    }, 
    {
        'variable': UnivariateExponential(rate=0.5), 
        'sample_size': 50
    }, 
]

common_variable = UnivariateGaussian(mean=1, std=1)
common_variable_sample = common_variable.sample(max([dict_['sample_size'] for dict_ in random_variable_meta]))

# sample from meta
data = [dict_['variable'].sample(dict_['sample_size']) + common_variable_sample[:dict_['sample_size']] for dict_ in random_variable_meta]

# calculate the mean 
def calculate_mean(meta: dict, sample_size: int = 500000) -> float: 
    sample_arr = np.zeros(sample_size)
    common_variable_sample = common_variable.sample(sample_size)
    for i in range(sample_size):
        sample_arr[i] = max([var['variable'].sample(1)[0] + common_variable_sample[i] for var in meta])

    return np.mean(sample_arr)
true_mean = calculate_mean(random_variable_meta)

In [78]:
print(f"Naive estimator: {naive_estimator(data)}")
print(f"Bootstrap estimator: {bootstrap_estimator(data)}")
print('True value: ', true_mean)

Naive estimator: 4.014787293255334
Bootstrap estimator: 7.553187547919147
True value:  7.72989813237055
